In [10]:
import os, subprocess
import json
import uproot
import awkward as ak
import numpy as np
from coffea import processor, util, hist
import pandas as pd
import pickle

In [11]:
lumis = {}
lumis['2016'] = 35.9
lumis['2017'] = 41.5
lumis['2018'] = 59.9

with open('xsec.json') as f:
  xs = json.load(f)

with open('pmap.json') as f:
  pmap = json.load(f)

In [12]:
year = '2017'
nfiles = len(subprocess.getoutput("ls infiles-split/"+year+"*.json").split())
outsum = processor.dict_accumulator()

ddbthr = 0.64

In [13]:
picklename = 'pickles/'+str(year)+'_templates.pkl'

# Read the histogram from the pickle file
ggf = pickle.load(open(picklename,'rb')).integrate('region','signal-ggf').integrate('systematic','nominal').integrate('mjj',overflow='allnan')
vbf = pickle.load(open(picklename,'rb')).integrate('region','signal-vbf').integrate('systematic','nominal').integrate('mjj',overflow='allnan')
mucr = pickle.load(open(picklename,'rb')).integrate('region','muoncontrol').integrate('systematic','nominal').integrate('mjj',overflow='allnan')

In [15]:
mc = ['QCD','Wjets','Zjets','Zjetsbb','VV','ttbar','singlet','ggF','VBF','WH','ZH','ttH']
cutflow = pd.DataFrame(columns=['ggffail','ggfpass','vbffail','vbfpass'],index=mc)

In [16]:
for p in mc:
    if 'Zjets' in p:
        continue
        
    cutflow['vbffail'][p] = vbf.sum('pt1','genflavor','msd1').integrate('ddb1',int_range=slice(0,ddbthr)).integrate('process',p).values()[()]
    cutflow['vbfpass'][p] = vbf.sum('pt1','genflavor','msd1').integrate('ddb1',int_range=slice(ddbthr,1)).integrate('process',p).values()[()]
    cutflow['ggffail'][p] = ggf.sum('pt1','genflavor','msd1').integrate('ddb1',int_range=slice(0,ddbthr)).integrate('process',p).values()[()]
    cutflow['ggfpass'][p] = ggf.sum('pt1','genflavor','msd1').integrate('ddb1',int_range=slice(ddbthr,1)).integrate('process',p).values()[()]
    
cutflow['vbffail']['Zjets'] = vbf.sum('pt1','msd1').integrate('genflavor',int_range=slice(0,3)).integrate('ddb1',int_range=slice(0,ddbthr)).integrate('process','Zjets').values()[()]
cutflow['vbfpass']['Zjets'] = vbf.sum('pt1','msd1').integrate('genflavor',int_range=slice(0,3)).integrate('ddb1',int_range=slice(ddbthr,1)).integrate('process','Zjets').values()[()]
cutflow['ggffail']['Zjets'] = ggf.sum('pt1','msd1').integrate('genflavor',int_range=slice(0,3)).integrate('ddb1',int_range=slice(0,ddbthr)).integrate('process','Zjets').values()[()]
cutflow['ggfpass']['Zjets'] = ggf.sum('pt1','msd1').integrate('genflavor',int_range=slice(0,3)).integrate('ddb1',int_range=slice(ddbthr,1)).integrate('process','Zjets').values()[()]
    
cutflow['vbffail']['Zjetsbb'] = vbf.sum('pt1','msd1').integrate('genflavor',int_range=slice(3,4)).integrate('ddb1',int_range=slice(0,ddbthr)).integrate('process','Zjets').values()[()]
cutflow['vbfpass']['Zjetsbb'] = vbf.sum('pt1','msd1').integrate('genflavor',int_range=slice(3,4)).integrate('ddb1',int_range=slice(ddbthr,1)).integrate('process','Zjets').values()[()]
cutflow['ggffail']['Zjetsbb'] = ggf.sum('pt1','msd1').integrate('genflavor',int_range=slice(3,4)).integrate('ddb1',int_range=slice(0,ddbthr)).integrate('process','Zjets').values()[()]
cutflow['ggfpass']['Zjetsbb'] = ggf.sum('pt1','msd1').integrate('genflavor',int_range=slice(3,4)).integrate('ddb1',int_range=slice(ddbthr,1)).integrate('process','Zjets').values()[()]

In [17]:
cutflow

,ggffail,ggfpass,vbffail,vbfpass
QCD,4969123.073044862,7048.843424234143,191272.4737530457,367.9309289746536
Wjets,100062.81312759504,191.15848720526034,3714.6121271489897,6.505457340086311
Zjets,38838.69465129493,144.85194845003187,1542.7920252423314,4.973155060998369
Zjetsbb,2523.7311480331146,875.7680657354996,92.91468597894756,32.20435950297917
VV,1477.4323355344754,20.044829970480947,47.458481803179346,0.19988878005639918
ttbar,19546.049073289516,192.52769956831676,656.7298415903025,8.293954053350038
singlet,3201.8315220548684,29.520529548093005,141.51773431517762,1.2334769596547828
ggF,57.81086529686069,23.973919442407105,1.7848465703232093,1.7242288272424096
VBF,14.378233787340376,6.556316637962743,10.303736436067032,5.214208034019105
WH,17.348527946433425,3.613774840621783,0.3760923851121675,0.17071608926925477


In [18]:
(cutflow.astype('int')).to_latex(buf=year+'/cutflow-final.tex')

In [19]:
total = cutflow['ggfpass'] + cutflow['ggffail'] + cutflow['vbfpass'] + cutflow['vbffail']
total

QCD        5167812.321151
Wjets       103975.089199
Zjets         40531.31178
Zjetsbb       3524.618259
VV            1545.135536
ttbar        20403.600569
singlet       3374.103263
ggF              85.29386
VBF             36.452495
WH              21.509111
ZH              16.865803
ttH             22.057941
dtype: object

In [20]:
(total['ggF']+total['VBF'])/sum(total)

2.2793187571790216e-05

In [21]:
a = vbf.integrate('pt1',overflow='allnan').integrate('msd1',overflow='allnan').integrate('ddb1',overflow='allnan').integrate('genflavor').integrate('process','QCD').values()
b = ggf.integrate('pt1',overflow='allnan').integrate('msd1',overflow='allnan').integrate('ddb1',overflow='allnan').integrate('genflavor').integrate('process','QCD').values()

In [22]:
a[()]+b[()]

5167812.32115112

In [23]:
vbf

<Hist (process,genflavor,pt1,msd1,ddb1) instance at 0x7f4ed8233b50>